# PDF Generation Examples
This notebook demonstrates creating PDF files with Python. It includes examples using ReportLab for direct canvas drawing and Platypus document generation, plus a second example using FPDF for custom header/footer pages.

## Simple ReportLab Canvas Example
This section shows how to use ReportLab's low-level canvas API to place text on a PDF page and save the file.

In [ ]:
from reportlab.pdfgen import canvas

# Create a ReportLab canvas object for the output PDF file.
c = canvas.Canvas(r".\output_reportlab.pdf")
# Draw a simple text string at the specified (x, y) coordinates on the page.
c.drawString(100, 750, "Hello, this is a test PDF.")
# Save the PDF to disk, finalizing the document.
c.save()

## ReportLab PDF with Table and Chart
This section uses ReportLab Platypus to create a styled PDF document including a title, table, spacing, and a vertical bar chart. The output file is saved as `output_reportlab.pdf`.

In [ ]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.graphics.charts.barcharts import VerticalBarChart
from reportlab.graphics.shapes import Drawing

# Create a document template for A4 page size.
doc = SimpleDocTemplate(r".\output_reportlab.pdf", pagesize=A4)
# Load default style definitions for paragraphs and titles.
styles = getSampleStyleSheet()

# Create a title paragraph for the PDF document.
title = Paragraph("Sample PDF with ReportLab", styles['Title'])

# Define the data sets to render in the table and chart.
data = {
    "Sample Data": [10, 20, 30, 40],
    "Another Dataset": [15, 25, 35, 45]
}

# Define the category labels for each data point.
categories = ["Q1", "Q2", "Q3", "Q4"]
columns = list(data.keys())

# Build table data rows with a header row plus values for each category.
table_data = [["Category"] + columns]
for i in range(len(categories)):
    row = [categories[i]] + [data[col][i] for col in columns]
    table_data.append(row)

# Create a table with fixed column widths and apply visual styling.
table = Table(table_data, colWidths=[100, 120, 120])
table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#1A365D")),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 12),
    ('BOTTOMPADDING', (0, 0), (-1, 0), 8),
    ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor("#F7FAFC")),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor("#CBD5E0")),
]))

# Configure a drawing object to host the bar chart.
drawing = Drawing(400, 200)
chart = VerticalBarChart()
chart.x = 50
chart.y = 20
chart.height = 160
chart.width = 300
chart.data = [data["Sample Data"], data["Another Dataset"]]
chart.categoryAxis.categoryNames = categories
chart.bars[0].fillColor = colors.HexColor("#3182CE")
chart.bars[1].fillColor = colors.HexColor("#DD6B20")

drawing.add(chart)

# Assemble the document story: title, spacing, table, spacing, and chart.
story = [
    title,
    Spacer(1, 20),
    table,
    Spacer(1, 30),
    drawing
]

# Build the PDF document from the story flow.
doc.build(story)
print("The PDF has been created successfully: output_reportlab.pdf")

The PDF has been created successfully: output_reportlab.pdf


## FPDF PDF with Custom Header and Footer
This section demonstrates creating a PDF with the FPDF library, including a custom header and footer class, text alignment, and font styling. The output file is saved as `output_fpdf.pdf`.

In [ ]:
from fpdf import FPDF
from PIL import ImageFont

# Define a custom PDF class that extends FPDF to add page headers and footers.
class PDF(FPDF):
    def header(self):
        self.set_font("Arial", "B", 12)
        self.cell(0, 10, "Sample PDF Header", ln=True, align="C")
        self.ln(10)

    def footer(self):
        self.set_y(-15)
        self.set_font("Arial", "I", 8)
        page_no = f"Page {self.page_no()}"
        self.cell(0, 10, page_no, align="C")

# Helper function to estimate the width of text using PIL font measurements.
def get_text_width(text, font):
    return font.getmask(text).getbbox()[2]/2.6

# Create a new PDF document in portrait orientation with mm units and Letter size.
pdf = PDF('P', 'mm', 'Letter')

pdf.set_title("Sample PDF with FPDF")
pdf.set_author("George_Samer")

# Enable automatic page breaks with a bottom margin.
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# Load a TrueType font for text width measurement.
font = ImageFont.truetype("arial.ttf", 12)

pdf.set_font("Arial", size=12)

text1 = "Hello, this is a test PDF created using FPDF."
# Add the first line of text to the PDF using the measured width.
pdf.cell(get_text_width(text1, font), 10, text1, ln=True)

pdf.set_font("Arial", "B", 14)

text2 = "This is a bold text."
pdf.cell(get_text_width(text2, font), 10, text2, ln=True)

# Save the FPDF output to disk.
pdf.output(r".\output_fpdf.pdf")

''